# 📕 Module 3 - SQL EXPERT (Enterprise & Production-Level)

## Course Overview
This module focuses on enterprise-grade SQL development, performance optimization, and production best practices. You'll learn techniques used by database professionals in large-scale systems.

### What You'll Master
* **Enterprise Database Design** - OLTP vs OLAP, dimensional modeling
* **Advanced Performance Tuning** - Query optimization, execution plans
* **Table Partitioning** - Managing large datasets efficiently
* **Stored Procedures & Functions** - Database programming
* **Triggers** - Automated database actions
* **Security & Compliance** - Row-level security, encryption, auditing
* **High Availability** - Backup, recovery, replication
* **Bulk Operations** - ETL patterns, large data loads
* **Monitoring & Troubleshooting** - Performance diagnostics
* **Advanced Integration** - External data sources, data lakes
* **Real-World Scenarios** - Production patterns and best practices

**Prerequisites:** Module 1 (Basic) & Module 2 (Advance)

**Duration:** 15-20 hours

**Target Audience:** Data engineers, database administrators, senior developers

---

## 1️⃣ Enterprise Database Architecture

### OLTP vs OLAP

#### OLTP (Online Transaction Processing)
**Purpose**: Handle day-to-day transactions
* **Characteristics**: Many users, frequent updates, normalized design
* **Examples**: Order processing, banking, e-commerce
* **Optimization**: Fast writes, transactional consistency
* **Schema**: Highly normalized (3NF)

#### OLAP (Online Analytical Processing)
**Purpose**: Support business intelligence and analytics
* **Characteristics**: Few users, read-heavy, denormalized design
* **Examples**: Data warehouses, reporting, analytics
* **Optimization**: Fast reads, aggregations
* **Schema**: Star/snowflake schemas

### Dimensional Modeling

#### Star Schema
Simplest dimensional model
* **Fact Table**: Center, contains metrics (sales, revenue)
* **Dimension Tables**: Surrounding, contain attributes (customer, product, date)
* **Benefits**: Simple queries, fast performance

```
    Dim_Customer
         |
    Fact_Sales --- Dim_Product
         |
    Dim_Date
```

#### Snowflake Schema
Normalized version of star schema
* Dimension tables are normalized into sub-dimensions
* **Benefits**: Less storage, but more complex queries

#### Galaxy/Constellation Schema
Multiple fact tables sharing dimension tables

### Design Principles
1. **Separate OLTP from OLAP** - Don't run reports on transactional databases
2. **Use appropriate schema** - Normalize for OLTP, denormalize for OLAP
3. **Plan for scale** - Consider partitioning, archiving strategies
4. **Index strategically** - Different strategies for different workloads

In [0]:
%sql
-- Example: Star Schema Design

-- Fact Table (center)
CREATE TABLE Fact_Sales (
  SaleID BIGINT PRIMARY KEY,
  DateKey INT,
  CustomerKey INT,
  ProductKey INT,
  StoreKey INT,
  Quantity INT,
  Amount DECIMAL(12,2),
  Cost DECIMAL(12,2),
  Profit DECIMAL(12,2),
  FOREIGN KEY (DateKey) REFERENCES Dim_Date(DateKey),
  FOREIGN KEY (CustomerKey) REFERENCES Dim_Customer(CustomerKey),
  FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
  FOREIGN KEY (StoreKey) REFERENCES Dim_Store(StoreKey)
);

-- Dimension Tables
CREATE TABLE Dim_Customer (
  CustomerKey INT PRIMARY KEY,
  CustomerID VARCHAR(50),
  CustomerName VARCHAR(100),
  City VARCHAR(50),
  State VARCHAR(50),
  Country VARCHAR(50),
  Segment VARCHAR(50)
);

CREATE TABLE Dim_Product (
  ProductKey INT PRIMARY KEY,
  ProductID VARCHAR(50),
  ProductName VARCHAR(100),
  Category VARCHAR(50),
  SubCategory VARCHAR(50),
  Brand VARCHAR(50),
  UnitPrice DECIMAL(10,2)
);

CREATE TABLE Dim_Date (
  DateKey INT PRIMARY KEY,
  Date DATE,
  Year INT,
  Quarter INT,
  Month INT,
  MonthName VARCHAR(20),
  Day INT,
  DayOfWeek INT,
  DayName VARCHAR(20),
  IsWeekend BOOLEAN,
  IsHoliday BOOLEAN
);

-- Analytical Query Example
-- SELECT 
--   d.Year,
--   d.Quarter,
--   p.Category,
--   SUM(f.Amount) AS TotalSales,
--   SUM(f.Profit) AS TotalProfit,
--   COUNT(DISTINCT f.CustomerKey) AS UniqueCustomers
-- FROM Fact_Sales f
-- JOIN Dim_Date d ON f.DateKey = d.DateKey
-- JOIN Dim_Product p ON f.ProductKey = p.ProductKey
-- WHERE d.Year = 2023
-- GROUP BY d.Year, d.Quarter, p.Category
-- ORDER BY d.Quarter, TotalSales DESC;

## 2️⃣ Query Performance & Optimization

### Execution Plans
An execution plan shows how the database engine executes a query.

#### Reading Execution Plans
```sql
-- Databricks/Spark SQL
EXPLAIN SELECT * FROM large_table WHERE category = 'Electronics';

-- SQL Server
SET SHOWPLAN_ALL ON;
SELECT * FROM large_table WHERE category = 'Electronics';

-- PostgreSQL
EXPLAIN ANALYZE SELECT * FROM large_table WHERE category = 'Electronics';
```

### Key Metrics
* **Estimated vs Actual Rows**: Large differences indicate stale statistics
* **Scan vs Seek**: Seek (using index) is faster than Scan (reading all rows)
* **Join Types**: Nested Loop, Hash Join, Merge Join
* **Cost**: Relative expense of operations

### Common Performance Issues

#### 1. Table Scans
**Problem**: Reading entire table instead of using index
**Solution**: Add appropriate indexes

#### 2. Missing Indexes
**Problem**: Slow WHERE, JOIN, ORDER BY operations
**Solution**: Create indexes on frequently queried columns

#### 3. Index Scans vs Index Seeks
**Problem**: Using index but still reading many rows
**Solution**: Make index more selective, add covering index

#### 4. Parameter Sniffing
**Problem**: Cached plan optimized for wrong parameter values
**Solution**: Use OPTION (RECOMPILE) or optimize for common values

#### 5. Implicit Conversions
**Problem**: Data type mismatches force conversions
**Solution**: Match data types in JOINs and WHERE clauses

### Optimization Techniques

#### 1. Index Optimization
```sql
-- Covering Index (includes all needed columns)
CREATE INDEX idx_orders_covering 
ON Orders(CustomerID, OrderDate) 
INCLUDE (Amount, Status);

-- Filtered Index (only for subset of data)
CREATE INDEX idx_active_orders 
ON Orders(OrderDate) 
WHERE Status = 'Active';
```

#### 2. Query Rewriting
```sql
-- Bad: Function on indexed column
SELECT * FROM Orders WHERE YEAR(OrderDate) = 2023;

-- Good: Preserve index usability
SELECT * FROM Orders 
WHERE OrderDate >= '2023-01-01' AND OrderDate < '2024-01-01';

-- Bad: OR conditions
SELECT * FROM Employees WHERE Department = 'IT' OR Department = 'HR';

-- Good: Use IN
SELECT * FROM Employees WHERE Department IN ('IT', 'HR');
```

#### 3. EXISTS vs IN
```sql
-- EXISTS (better for large subqueries)
SELECT * FROM Customers c
WHERE EXISTS (
  SELECT 1 FROM Orders o WHERE o.CustomerID = c.CustomerID
);

-- IN (better for small, static lists)
SELECT * FROM Customers WHERE City IN ('New York', 'Los Angeles');
```

#### 4. Join Order
```sql
-- Filter early to reduce rows
SELECT e.FirstName, d.DepartmentName
FROM (
  SELECT * FROM Employees WHERE Salary > 100000
) e
JOIN Departments d ON e.DepartmentID = d.DepartmentID;
```

In [0]:
%sql
-- Example 1: Optimize with covering index
-- Before: Slow query
SELECT CustomerID, OrderDate, Amount
FROM Orders
WHERE CustomerID = 1001
ORDER BY OrderDate DESC;

-- Create covering index
-- CREATE INDEX idx_orders_customer_covering
-- ON Orders(CustomerID, OrderDate DESC)
-- INCLUDE (Amount);

-- Example 2: Avoid function on indexed column
-- Bad
-- SELECT * FROM Employees WHERE UPPER(LastName) = 'SMITH';

-- Good: Use indexed column directly
SELECT * FROM Employees WHERE LastName = 'Smith';

-- Example 3: Use EXISTS for existence checks
-- Efficient existence check
SELECT c.CustomerName
FROM Customers c
WHERE EXISTS (
  SELECT 1 FROM Orders o 
  WHERE o.CustomerID = c.CustomerID AND o.Amount > 1000
);

## 3️⃣ Table Partitioning

Partitioning divides large tables into smaller, manageable pieces while appearing as a single table.

### Benefits
* **Performance**: Query only relevant partitions
* **Manageability**: Easier maintenance, archiving
* **Parallel processing**: Different partitions processed simultaneously
* **Data lifecycle**: Easy to drop old partitions

### Partitioning Strategies

#### 1. Range Partitioning
Partition by value ranges (common for dates, IDs)
```sql
CREATE TABLE Sales (
  SaleID BIGINT,
  SaleDate DATE,
  Amount DECIMAL(10,2)
)
PARTITIONED BY (YEAR(SaleDate));
```

#### 2. List Partitioning
Partition by discrete values
```sql
PARTITIONED BY (Region)  -- 'North', 'South', 'East', 'West'
```

#### 3. Hash Partitioning
Distribute data evenly across partitions using hash function

#### 4. Composite Partitioning
Combine strategies (e.g., range by year, hash by customer)

### Best Practices
* Partition on frequently filtered columns
* Keep partition count reasonable (100-1000 partitions typical)
* Align partitions with query patterns
* Use partition pruning (filter on partition key)
* Consider storage costs vs performance gains

In [0]:
%sql
-- Databricks: Create partitioned table
CREATE TABLE IF NOT EXISTS Sales_Partitioned (
  SaleID BIGINT,
  CustomerID INT,
  ProductID INT,
  Amount DECIMAL(10,2),
  SaleDate DATE,
  Region VARCHAR(50)
)
PARTITIONED BY (year INT, month INT);

-- Insert data with partition values
-- INSERT INTO Sales_Partitioned 
-- PARTITION (year = 2023, month = 1)
-- SELECT SaleID, CustomerID, ProductID, Amount, SaleDate, Region
-- FROM Sales_Source
-- WHERE YEAR(SaleDate) = 2023 AND MONTH(SaleDate) = 1;

-- Query with partition pruning (efficient)
-- SELECT * FROM Sales_Partitioned
-- WHERE year = 2023 AND month BETWEEN 1 AND 3;

-- Show partitions
-- SHOW PARTITIONS Sales_Partitioned;

-- Drop old partition (for archiving)
-- ALTER TABLE Sales_Partitioned DROP PARTITION (year = 2020);

## 4️⃣ Stored Procedures & Functions

Stored procedures encapsulate business logic in reusable, compiled code.

### Benefits
* **Performance**: Pre-compiled, execution plan cached
* **Security**: Control access to underlying tables
* **Maintainability**: Centralized business logic
* **Reduced network traffic**: Single call executes multiple operations
* **Transaction management**: Ensure data consistency

### Stored Procedure Syntax
```sql
CREATE PROCEDURE procedure_name (
  @param1 datatype,
  @param2 datatype OUTPUT
)
AS
BEGIN
  -- SQL statements
  -- SET @param2 = calculated_value
END;

-- Execute
EXEC procedure_name @param1 = value;
```

### User-Defined Functions (UDF)

#### Scalar Functions
Return single value
```sql
CREATE FUNCTION CalculateTax(@Amount DECIMAL(10,2))
RETURNS DECIMAL(10,2)
AS
BEGIN
  RETURN @Amount * 0.15;
END;
```

#### Table-Valued Functions
Return table
```sql
CREATE FUNCTION GetEmployeesByDept(@DeptID INT)
RETURNS TABLE
AS
RETURN (
  SELECT * FROM Employees WHERE DepartmentID = @DeptID
);
```

### Best Practices
* Use meaningful parameter names
* Include error handling (TRY/CATCH)
* Document parameters and return values
* Keep procedures focused (single responsibility)
* Use transactions for data modifications

In [0]:
%sql
-- Example: Stored Procedure for transferring funds
-- Note: Syntax varies by database (SQL Server example)

/*
CREATE PROCEDURE TransferFunds
  @FromAccountID INT,
  @ToAccountID INT,
  @Amount DECIMAL(10,2),
  @Status VARCHAR(50) OUTPUT
AS
BEGIN
  SET NOCOUNT ON;
  DECLARE @FromBalance DECIMAL(10,2);
  
  BEGIN TRY
    BEGIN TRANSACTION;
    
    -- Check source account balance
    SELECT @FromBalance = Balance FROM Accounts WHERE AccountID = @FromAccountID;
    
    IF @FromBalance < @Amount
    BEGIN
      SET @Status = 'Insufficient funds';
      ROLLBACK TRANSACTION;
      RETURN;
    END
    
    -- Debit source account
    UPDATE Accounts SET Balance = Balance - @Amount WHERE AccountID = @FromAccountID;
    
    -- Credit destination account
    UPDATE Accounts SET Balance = Balance + @Amount WHERE AccountID = @ToAccountID;
    
    COMMIT TRANSACTION;
    SET @Status = 'Success';
    
  END TRY
  BEGIN CATCH
    IF @@TRANCOUNT > 0
      ROLLBACK TRANSACTION;
    
    SET @Status = 'Error: ' + ERROR_MESSAGE();
  END CATCH;
END;

-- Execute
DECLARE @Result VARCHAR(50);
EXEC TransferFunds @FromAccountID = 1, @ToAccountID = 2, @Amount = 500, @Status = @Result OUTPUT;
SELECT @Result AS TransferStatus;
*/

-- Databricks SQL: Create function
CREATE OR REPLACE FUNCTION CalculateDiscount(price DOUBLE, discount_pct DOUBLE)
RETURNS DOUBLE
RETURN price * (1 - discount_pct / 100);

-- Use function
-- SELECT ProductName, Price, CalculateDiscount(Price, 10) AS DiscountedPrice FROM Products;

## 5️⃣ Triggers - Automated Actions

Triggers automatically execute in response to specific events.

### Types of Triggers

#### 1. DML Triggers
Fire on INSERT, UPDATE, DELETE
```sql
CREATE TRIGGER trigger_name
ON table_name
AFTER INSERT, UPDATE
AS
BEGIN
  -- Actions to perform
END;
```

#### 2. DDL Triggers
Fire on CREATE, ALTER, DROP

#### 3. LOGON Triggers
Fire on user login events

### Trigger Timing
* **BEFORE**: Execute before the operation (validation)
* **AFTER**: Execute after the operation (auditing, cascading updates)
* **INSTEAD OF**: Replace the operation (views)

### Common Use Cases
1. **Audit trails**: Log who changed what and when
2. **Data validation**: Enforce complex business rules
3. **Cascading updates**: Update related tables automatically
4. **Derived columns**: Calculate and populate values
5. **Prevent operations**: Block certain actions

### Special Tables in Triggers
* **INSERTED**: Contains new/updated rows
* **DELETED**: Contains old/deleted rows

### Best Practices
⚠️ **Use triggers sparingly!**
* Can impact performance
* Hidden logic (hard to debug)
* Cascading triggers can be complex
* Consider alternatives (constraints, stored procedures)

✅ **When to use triggers:**
* Audit logging
* Complex referential integrity
* Maintaining summary tables
* Enforcing business rules not possible with constraints

In [0]:
%sql
-- Example 1: Audit trigger (SQL Server syntax)
/*
CREATE TABLE EmployeeAudit (
  AuditID INT IDENTITY PRIMARY KEY,
  EmployeeID INT,
  Action VARCHAR(10),
  OldSalary DECIMAL(10,2),
  NewSalary DECIMAL(10,2),
  ChangedBy VARCHAR(100),
  ChangedDate DATETIME DEFAULT GETDATE()
);

CREATE TRIGGER trg_Employee_Salary_Audit
ON Employees
AFTER UPDATE
AS
BEGIN
  IF UPDATE(Salary)
  BEGIN
    INSERT INTO EmployeeAudit (EmployeeID, Action, OldSalary, NewSalary, ChangedBy)
    SELECT 
      i.EmployeeID,
      'UPDATE',
      d.Salary,
      i.Salary,
      SYSTEM_USER
    FROM INSERTED i
    INNER JOIN DELETED d ON i.EmployeeID = d.EmployeeID
    WHERE i.Salary <> d.Salary;
  END
END;
*/

-- Example 2: Validation trigger
/*
CREATE TRIGGER trg_Validate_Salary
ON Employees
FOR INSERT, UPDATE
AS
BEGIN
  IF EXISTS (SELECT * FROM INSERTED WHERE Salary < 0 OR Salary > 1000000)
  BEGIN
    RAISERROR ('Salary must be between 0 and 1,000,000', 16, 1);
    ROLLBACK TRANSACTION;
  END
END;
*/

-- Example 3: Instead of trigger on view
/*
CREATE TRIGGER trg_UpdateEmployeeView
ON vw_EmployeeSummary
INSTEAD OF UPDATE
AS
BEGIN
  UPDATE Employees
  SET Salary = i.Salary
  FROM Employees e
  INNER JOIN INSERTED i ON e.EmployeeID = i.EmployeeID;
END;
*/

SELECT 'Triggers examples above (commented - syntax varies by database)' AS Note;

## 6️⃣ Security & Compliance

### Row-Level Security (RLS)
Restrict which rows users can access based on their identity.

```sql
-- Create security policy
CREATE SECURITY POLICY SalesFilter
ADD FILTER PREDICATE dbo.fn_SecurityPredicate(SalesPersonID)
ON Sales
WITH (STATE = ON);

-- Security function
CREATE FUNCTION fn_SecurityPredicate(@SalesPersonID INT)
RETURNS TABLE
WITH SCHEMABINDING
AS
RETURN SELECT 1 AS result
WHERE @SalesPersonID = CAST(SESSION_CONTEXT(N'SalesPersonID') AS INT);
```

### Column-Level Security
Mask sensitive data from unauthorized users.

```sql
-- Dynamic Data Masking (SQL Server)
ALTER TABLE Customers
ALTER COLUMN Email ADD MASKED WITH (FUNCTION = 'email()');

ALTER TABLE Customers
ALTER COLUMN Phone ADD MASKED WITH (FUNCTION = 'partial(0,"XXX-XXX-",4)');

ALTER TABLE Customers
ALTER COLUMN CreditCard ADD MASKED WITH (FUNCTION = 'partial(0,"XXXX-XXXX-XXXX-",4)');
```

### Encryption

#### Transparent Data Encryption (TDE)
* Encrypts entire database at rest
* No application changes required

#### Column-Level Encryption
* Encrypt specific sensitive columns
* Application must handle encryption/decryption

#### Always Encrypted
* Data encrypted in transit and at rest
* SQL Server never sees unencrypted data

### Access Control

```sql
-- Create roles
CREATE ROLE DataAnalyst;
CREATE ROLE DataEngineer;

-- Grant permissions to roles
GRANT SELECT ON SCHEMA::Sales TO DataAnalyst;
GRANT SELECT, INSERT, UPDATE ON SCHEMA::Sales TO DataEngineer;

-- Add users to roles
ALTER ROLE DataAnalyst ADD MEMBER john_doe;
ALTER ROLE DataEngineer ADD MEMBER jane_smith;

-- Revoke permissions
REVOKE UPDATE ON Employees FROM DataAnalyst;
```

### Audit Logging

```sql
-- Enable auditing (SQL Server)
CREATE SERVER AUDIT CompanyAudit
TO FILE (FILEPATH = 'C:\Audit\');

CREATE DATABASE AUDIT SPECIFICATION DB_Audit
FOR SERVER AUDIT CompanyAudit
ADD (SELECT, INSERT, UPDATE, DELETE ON DATABASE::CompanyDB BY public);

ALTER SERVER AUDIT CompanyAudit WITH (STATE = ON);
```

### Best Practices
1. **Principle of least privilege**: Grant minimum necessary permissions
2. **Use roles**: Manage permissions at role level, not user level
3. **Audit access**: Log who accesses sensitive data
4. **Encrypt sensitive data**: Both at rest and in transit
5. **Regular reviews**: Audit user permissions quarterly
6. **Separate duties**: DBA, developer, auditor roles separate
7. **Use service accounts**: Applications shouldn't use user credentials

## 7️⃣ Bulk Operations & ETL Patterns

### Bulk Insert Operations

Loading large volumes of data efficiently.

#### SQL Server BULK INSERT
```sql
BULK INSERT Sales
FROM 'C:\Data\sales_2023.csv'
WITH (
  FIELDTERMINATOR = ',',
  ROWTERMINATOR = '\n',
  FIRSTROW = 2,
  TABLOCK,
  BATCHSIZE = 10000
);
```

#### Databricks COPY INTO
```sql
COPY INTO target_table
FROM '/mnt/data/sales/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

### ETL Best Practices

#### 1. Staging Tables
Load raw data first, then transform
```sql
-- Stage 1: Load to staging
CREATE TABLE Staging_Sales AS
SELECT * FROM read_files('/data/sales/*.csv');

-- Stage 2: Transform and load to production
INSERT INTO Production_Sales
SELECT 
  SaleID,
  UPPER(TRIM(CustomerName)) AS CustomerName,
  CAST(SaleDate AS DATE) AS SaleDate,
  Amount,
  CURRENT_TIMESTAMP() AS LoadedAt
FROM Staging_Sales
WHERE Amount > 0;  -- Data quality check
```

#### 2. Incremental Loads
```sql
-- Track last load
CREATE TABLE ETL_Control (
  TableName VARCHAR(100),
  LastLoadDate TIMESTAMP
);

-- Load only new data
INSERT INTO Target_Table
SELECT *
FROM Source_Table
WHERE ModifiedDate > (
  SELECT LastLoadDate FROM ETL_Control WHERE TableName = 'Target_Table'
);

-- Update control table
UPDATE ETL_Control
SET LastLoadDate = CURRENT_TIMESTAMP()
WHERE TableName = 'Target_Table';
```

#### 3. Merge/Upsert Pattern
```sql
MERGE INTO Target t
USING Source s
ON t.ID = s.ID
WHEN MATCHED THEN
  UPDATE SET t.Value = s.Value, t.ModifiedDate = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (ID, Value, CreatedDate) VALUES (s.ID, s.Value, CURRENT_TIMESTAMP());
```

### Performance Optimization for Bulk Loads

1. **Disable indexes** during load, rebuild after
2. **Use TABLOCK** for minimal logging
3. **Batch operations** in chunks (10K-100K rows)
4. **Parallel loading** when possible
5. **Drop constraints**, reload, re-enable
6. **Use staging tables** for transformations
7. **Compress data files** before loading

In [0]:
%sql
-- Example: Efficient data loading pattern

-- Step 1: Create staging table (no indexes, no constraints)
CREATE TABLE Staging_Orders (
  OrderID BIGINT,
  CustomerID INT,
  OrderDate STRING,  -- Load as string first
  Amount STRING,
  Status STRING
);

-- Step 2: Bulk load raw data
-- COPY INTO Staging_Orders
-- FROM '/mnt/data/orders/'
-- FILEFORMAT = CSV;

-- Step 3: Data quality and transformation
CREATE TABLE Production_Orders AS
SELECT 
  CAST(OrderID AS BIGINT) AS OrderID,
  CAST(CustomerID AS INT) AS CustomerID,
  CAST(OrderDate AS DATE) AS OrderDate,
  CAST(Amount AS DECIMAL(10,2)) AS Amount,
  UPPER(TRIM(Status)) AS Status,
  CURRENT_TIMESTAMP() AS LoadedAt
FROM Staging_Orders
WHERE 
  OrderID IS NOT NULL
  AND CustomerID IS NOT NULL
  AND TRY_CAST(Amount AS DECIMAL(10,2)) IS NOT NULL  -- Validate numeric
  AND Status IN ('Pending', 'Completed', 'Cancelled');  -- Validate values

-- Step 4: Create indexes after load
-- CREATE INDEX idx_orders_customer ON Production_Orders(CustomerID);
-- CREATE INDEX idx_orders_date ON Production_Orders(OrderDate);

-- Step 5: Clean up staging
-- DROP TABLE Staging_Orders;

## 8️⃣ High Availability & Disaster Recovery

### Backup Strategies

#### Types of Backups

**1. Full Backup**
* Complete copy of database
* Longest duration, largest size
* Baseline for recovery

**2. Differential Backup**
* Changes since last full backup
* Faster than full, larger than incremental

**3. Transaction Log Backup**
* Changes since last log backup
* Point-in-time recovery
* Smallest, fastest

#### Backup Schedule Example
```
- Full backup: Sunday 2 AM (weekly)
- Differential backup: Daily 2 AM (except Sunday)
- Log backup: Every 15 minutes (24/7)
```

### Backup Commands

```sql
-- SQL Server: Full backup
BACKUP DATABASE CompanyDB
TO DISK = 'C:\Backups\CompanyDB_Full.bak'
WITH COMPRESSION, STATS = 10;

-- Differential backup
BACKUP DATABASE CompanyDB
TO DISK = 'C:\Backups\CompanyDB_Diff.bak'
WITH DIFFERENTIAL, COMPRESSION;

-- Transaction log backup
BACKUP LOG CompanyDB
TO DISK = 'C:\Backups\CompanyDB_Log.trn'
WITH COMPRESSION;
```

### Recovery Scenarios

#### Scenario 1: Complete Database Loss
```sql
-- Restore full backup
RESTORE DATABASE CompanyDB
FROM DISK = 'C:\Backups\CompanyDB_Full.bak'
WITH NORECOVERY;

-- Restore latest differential
RESTORE DATABASE CompanyDB
FROM DISK = 'C:\Backups\CompanyDB_Diff.bak'
WITH NORECOVERY;

-- Restore all log backups
RESTORE LOG CompanyDB
FROM DISK = 'C:\Backups\CompanyDB_Log_01.trn'
WITH RECOVERY;  -- Last one uses RECOVERY
```

#### Scenario 2: Point-in-Time Recovery
```sql
RESTORE DATABASE CompanyDB
FROM DISK = 'C:\Backups\CompanyDB_Full.bak'
WITH NORECOVERY;

RESTORE LOG CompanyDB
FROM DISK = 'C:\Backups\CompanyDB_Log.trn'
WITH STOPAT = '2023-09-21 14:30:00', RECOVERY;
```

### High Availability Solutions

#### 1. Database Mirroring
* Automatic failover to mirror server
* Synchronous or asynchronous

#### 2. Always On Availability Groups
* Multiple replicas
* Read-only secondaries
* Automatic failover

#### 3. Replication
* Transactional replication
* Merge replication
* Snapshot replication

#### 4. Log Shipping
* Simple, cost-effective
* Automatic backup and restore

### RTO & RPO

* **RTO (Recovery Time Objective)**: How long can you be down?
* **RPO (Recovery Point Objective)**: How much data loss is acceptable?

| Solution | RTO | RPO |
|----------|-----|-----|
| Always On AG | Seconds-Minutes | Near zero |
| Database Mirroring | Seconds | Near zero |
| Log Shipping | Minutes-Hours | Minutes |
| Backup/Restore | Hours | Last backup |

### Best Practices
1. **Test restores regularly** - Backups are worthless if they don't restore
2. **Store backups offsite** - Protect against site disasters
3. **Encrypt backups** - Secure sensitive data
4. **Monitor backup jobs** - Alert on failures
5. **Document procedures** - Recovery runbooks
6. **Practice drills** - Test disaster recovery annually

## 9️⃣ Monitoring & Troubleshooting

### Dynamic Management Views (DMVs)

DMVs provide real-time insights into database performance.

#### Most Expensive Queries
```sql
-- SQL Server: Top 10 CPU consumers
SELECT TOP 10
  total_worker_time/execution_count AS AvgCPU,
  total_elapsed_time/execution_count AS AvgDuration,
  execution_count,
  SUBSTRING(text, (statement_start_offset/2)+1,
    ((CASE statement_end_offset
      WHEN -1 THEN DATALENGTH(text)
      ELSE statement_end_offset END
      - statement_start_offset)/2) + 1) AS QueryText
FROM sys.dm_exec_query_stats
CROSS APPLY sys.dm_exec_sql_text(sql_handle)
ORDER BY AvgCPU DESC;
```

#### Long-Running Queries
```sql
SELECT 
  session_id,
  status,
  command,
  blocking_session_id,
  wait_type,
  wait_time,
  cpu_time,
  total_elapsed_time/1000 AS elapsed_seconds,
  text AS query_text
FROM sys.dm_exec_requests
CROSS APPLY sys.dm_exec_sql_text(sql_handle)
WHERE session_id > 50
ORDER BY total_elapsed_time DESC;
```

#### Index Usage Statistics
```sql
SELECT 
  OBJECT_NAME(s.object_id) AS TableName,
  i.name AS IndexName,
  s.user_seeks,
  s.user_scans,
  s.user_lookups,
  s.user_updates,
  s.last_user_seek,
  s.last_user_scan
FROM sys.dm_db_index_usage_stats s
JOIN sys.indexes i ON s.object_id = i.object_id AND s.index_id = i.index_id
WHERE OBJECTPROPERTY(s.object_id, 'IsUserTable') = 1
ORDER BY s.user_seeks + s.user_scans + s.user_lookups DESC;
```

#### Missing Indexes
```sql
SELECT 
  d.statement AS TableName,
  s.avg_user_impact * (s.user_seeks + s.user_scans) AS Impact,
  d.equality_columns,
  d.inequality_columns,
  d.included_columns
FROM sys.dm_db_missing_index_details d
JOIN sys.dm_db_missing_index_stats s
  ON d.index_handle = s.index_handle
ORDER BY Impact DESC;
```

### Wait Statistics

```sql
-- What is the database waiting for?
SELECT TOP 20
  wait_type,
  wait_time_ms / 1000.0 AS wait_time_s,
  waiting_tasks_count,
  (wait_time_ms / waiting_tasks_count) AS avg_wait_ms
FROM sys.dm_os_wait_stats
WHERE wait_type NOT LIKE '%SLEEP%'
ORDER BY wait_time_ms DESC;
```

### Blocking & Deadlocks

```sql
-- Find blocking sessions
SELECT 
  blocking.session_id AS BlockingSessionId,
  blocked.session_id AS BlockedSessionId,
  blocking_text.text AS BlockingQuery,
  blocked_text.text AS BlockedQuery,
  blocked.wait_time,
  blocked.wait_type
FROM sys.dm_exec_requests blocked
JOIN sys.dm_exec_requests blocking
  ON blocked.blocking_session_id = blocking.session_id
CROSS APPLY sys.dm_exec_sql_text(blocking.sql_handle) blocking_text
CROSS APPLY sys.dm_exec_sql_text(blocked.sql_handle) blocked_text;
```

### Performance Counters

Key metrics to monitor:
* **CPU utilization**: < 80% sustained
* **Memory pressure**: Page life expectancy > 300 seconds
* **Disk latency**: < 10ms read, < 20ms write
* **Batch requests/sec**: Throughput indicator
* **Compilations/sec**: Should be low relative to batch requests
* **Buffer cache hit ratio**: > 95%
* **Deadlocks/sec**: Should be near zero

### Query Store

Automatic query performance history (SQL Server 2016+)

```sql
-- Enable Query Store
ALTER DATABASE CompanyDB
SET QUERY_STORE = ON;

-- Find regressed queries
SELECT 
  q.query_id,
  qt.query_sql_text,
  rs.avg_duration,
  rs.avg_cpu_time,
  rs.execution_count
FROM sys.query_store_query q
JOIN sys.query_store_query_text qt ON q.query_text_id = qt.query_text_id
JOIN sys.query_store_plan p ON q.query_id = p.query_id
JOIN sys.query_store_runtime_stats rs ON p.plan_id = rs.plan_id
ORDER BY rs.avg_duration DESC;
```

### Troubleshooting Checklist

1. **Check execution plan** - Table scans? Missing indexes?
2. **Review wait statistics** - What's causing delays?
3. **Analyze blocking** - Concurrency issues?
4. **Update statistics** - Stale stats cause poor plans
5. **Rebuild fragmented indexes** - > 30% fragmentation
6. **Review parameter sniffing** - Wrong cached plan?
7. **Check tempdb contention** - Allocation issues?
8. **Monitor disk I/O** - Storage bottleneck?
9. **Review query patterns** - N+1 queries? Missing JOINs?

## 🎯 Module 3 - Expert Practice Exercises

### Exercise Set 1: Architecture & Design
1. Design a star schema for an e-commerce business with facts: Sales, Returns
2. Choose appropriate partitioning strategy for a 10TB sales table
3. Design a slowly changing dimension (Type 2) for customer data
4. Calculate storage requirements for 5 years of sales data

### Exercise Set 2: Performance Optimization
1. Analyze execution plan for a slow query and identify bottlenecks
2. Create appropriate indexes for common query patterns
3. Rewrite a query with functions on indexed columns
4. Optimize a query using window functions instead of self-joins
5. Identify and resolve parameter sniffing issues

### Exercise Set 3: Advanced Programming
1. Create a stored procedure for month-end closing with error handling
2. Write a trigger to maintain an audit trail of salary changes
3. Implement a user-defined function to calculate business days
4. Create a table-valued function for fiscal calendar conversion

### Exercise Set 4: ETL & Data Management
1. Design an incremental load process for large fact tables
2. Implement a MERGE statement for SCD Type 2
3. Create a staging-to-production ETL pipeline with validation
4. Handle duplicate records in bulk imports

### Exercise Set 5: Security & Compliance
1. Implement row-level security for multi-tenant database
2. Design a role-based access control scheme
3. Create masked views for PII data
4. Implement audit logging for sensitive table access

### Exercise Set 6: Monitoring & Troubleshooting
1. Write DMV queries to find top 10 slowest queries
2. Identify missing indexes using DMVs
3. Analyze wait statistics to find bottlenecks
4. Create alerts for long-running queries

### Exercise Set 7: HA & DR
1. Design a backup strategy for a 24/7 OLTP database
2. Calculate RPO and RTO for different HA solutions
3. Create a disaster recovery runbook
4. Plan a zero-downtime database upgrade

In [0]:
%sql
-- Use this cell to practice expert-level SQL

-- Example: Create dimension table with SCD Type 2
CREATE TABLE Dim_Customer_SCD2 (
  CustomerKey INT PRIMARY KEY,
  CustomerID VARCHAR(50),
  CustomerName VARCHAR(100),
  City VARCHAR(50),
  State VARCHAR(50),
  -- SCD Type 2 columns
  EffectiveDate DATE,
  EndDate DATE,
  IsCurrent BOOLEAN,
  -- Audit columns
  CreatedDate TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Your solutions here:



## 🎓 Module 3 - EXPERT Summary

Congratulations on completing SQL EXPERT! You now possess enterprise-level database skills.

### ✅ Expert Skills Mastered

#### Enterprise Architecture
* **OLTP vs OLAP design** - Transaction vs analytics optimization
* **Dimensional modeling** - Star/snowflake schemas
* **Data warehousing** - Fact/dimension tables
* **Scalability patterns** - Partitioning, sharding

#### Performance Mastery
* **Query optimization** - Execution plan analysis
* **Indexing strategies** - Covering, filtered, composite indexes
* **Query rewriting** - Avoiding anti-patterns
* **Statistics management** - Keep optimizer informed

#### Advanced Programming
* **Stored procedures** - Encapsulated business logic
* **User-defined functions** - Reusable calculations
* **Triggers** - Automated actions
* **Error handling** - Robust TRY/CATCH patterns

#### Data Engineering
* **Bulk operations** - COPY INTO, BULK INSERT
* **ETL patterns** - Staging, incremental loads, MERGE
* **Data quality** - Validation, cleansing
* **Pipeline design** - Efficient data movement

#### Production Operations
* **Security** - Row-level, column masking, encryption
* **High availability** - Backups, failover, replication
* **Monitoring** - DMVs, performance counters
* **Troubleshooting** - Wait stats, blocking, deadlocks

### 🛠️ Production Best Practices

#### Query Development
1. **Always test with production-like data volumes**
2. **Review execution plans** before deploying
3. **Use parameterized queries** to prevent SQL injection
4. **Avoid SELECT *** in production code
5. **Test with realistic concurrency**

#### Index Management
1. **Monitor index usage** - Drop unused indexes
2. **Rebuild fragmented indexes** regularly
3. **Balance read vs write** performance
4. **Include columns** for covering indexes
5. **Filter indexes** for subset queries

#### Performance
1. **Set-based operations** over cursors
2. **Batch large operations** (avoid single-row loops)
3. **Partition large tables** by access patterns
4. **Cache frequently used data** appropriately
5. **Monitor and tune** continuously

#### Security
1. **Principle of least privilege** always
2. **Encrypt sensitive data** at rest and in transit
3. **Audit critical operations** comprehensively
4. **Use roles** not individual grants
5. **Review permissions** quarterly

#### Reliability
1. **Test backups** by restoring regularly
2. **Automate** repetitive tasks
3. **Document** runbooks and procedures
4. **Monitor** proactively, don't react
5. **Version control** all database code

### 📈 Career Path

With these expert skills, you're qualified for:
* **Senior Database Developer**
* **Data Engineer**
* **Database Administrator (DBA)**
* **Data Architect**
* **BI/Analytics Engineer**
* **Performance Tuning Specialist**

### 📚 Continued Learning

1. **Cloud databases**: Azure SQL, AWS RDS, Google BigQuery
2. **NoSQL databases**: MongoDB, Cassandra, DynamoDB
3. **Data lakes**: Delta Lake, Iceberg, Hudi
4. **Streaming**: Kafka, Spark Structured Streaming
5. **Machine Learning**: SQL-based ML, feature engineering
6. **DevOps**: CI/CD for databases, infrastructure as code

### 🏆 You've Completed the Full SQL Learning Path!

**Module 1 (BASIC)** → **Module 2 (ADVANCE)** → **Module 3 (EXPERT)** ✓

You now have comprehensive SQL expertise from foundations to enterprise-level production systems. Keep practicing, stay current with new features, and apply these skills to solve real-world problems!

**Congratulations! 🎉**